# Flight Maneuver Classification

Classify flight maneuvers into 3 classes based on time-series accelerometer data (x, y, z measurements).

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report, f1_score,
    precision_recall_fscore_support, roc_auc_score
)
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Load Data

In [ ]:
# Load training data
train_df = pd.read_csv('../data/train_set.csv')
test_df = pd.read_csv('../data/test_set.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"\nTrain columns: {train_df.columns.tolist()}")
print(f"Test columns: {test_df.columns.tolist()}")

In [ ]:
# Inspect first rows
print("Training data sample:")
print(train_df.head())
print(f"\nLabel distribution:\n{train_df['label'].value_counts().sort_index()}")

## 3. EDA & Data Visualization

In [ ]:
# Data info
print("Missing values:")
print(train_df.isnull().sum())
print(f"\nUnique maneuvers: {train_df['maneuver_Id'].nunique()}")
print(f"Measurements per maneuver: {train_df.groupby('maneuver_Id').size().describe()}")

In [ ]:
# Visualization: Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train_df['label'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Label Distribution (Raw Counts)')
axes[0].set_ylabel('Count')

(train_df['label'].value_counts(normalize=True).sort_index() * 100).plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Label Distribution (Percentage)')
axes[1].set_ylabel('Percentage')
plt.tight_layout()
plt.show()

In [ ]:
# Visualization: Measurement distributions by label
fig, axes = plt.subplots(3, 1, figsize=(12, 9))
measurements = ['measurement_x', 'measurement_y', 'measurement_z']
for idx, m in enumerate(measurements):
    for label in sorted(train_df['label'].unique()):
        data = train_df[train_df['label'] == label][m]
        axes[idx].hist(data, alpha=0.6, bins=50, label=f'Class {label}')
    axes[idx].set_title(f'Distribution of {m}')
    axes[idx].set_xlabel(m)
    axes[idx].set_ylabel('Frequency')
    axes[idx].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots by label
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for idx, m in enumerate(measurements):
    train_df.boxplot(column=m, by='label', ax=axes[idx])
    axes[idx].set_title(f'{m} by Label')
    axes[idx].set_xlabel('Label')
plt.suptitle('')
plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
def extract_features(group):
    """Extract statistical features from time-series data for a single maneuver."""
    features = {}
    
    # Basic statistics for each measurement
    for col in ['measurement_x', 'measurement_y', 'measurement_z']:
        features[f'{col}_mean'] = group[col].mean()
        features[f'{col}_std'] = group[col].std()
        features[f'{col}_min'] = group[col].min()
        features[f'{col}_max'] = group[col].max()
        features[f'{col}_median'] = group[col].median()
        features[f'{col}_q25'] = group[col].quantile(0.25)
        features[f'{col}_q75'] = group[col].quantile(0.75)
        features[f'{col}_range'] = group[col].max() - group[col].min()
        features[f'{col}_skew'] = group[col].skew()
        features[f'{col}_kurtosis'] = group[col].kurtosis()
    
    # Magnitude of acceleration vector
    group['magnitude'] = np.sqrt(
        group['measurement_x']**2 + 
        group['measurement_y']**2 + 
        group['measurement_z']**2
    )
    features['magnitude_mean'] = group['magnitude'].mean()
    features['magnitude_std'] = group['magnitude'].std()
    features['magnitude_max'] = group['magnitude'].max()
    
    # Pairwise correlations
    corr_matrix = group[['measurement_x', 'measurement_y', 'measurement_z']].corr()
    features['corr_xy'] = corr_matrix.loc['measurement_x', 'measurement_y']
    features['corr_xz'] = corr_matrix.loc['measurement_x', 'measurement_z']
    features['corr_yz'] = corr_matrix.loc['measurement_y', 'measurement_z']
    
    # Rate of change (derivatives)
    for col in ['measurement_x', 'measurement_y', 'measurement_z']:
        diff = group[col].diff().dropna()
        if len(diff) > 0:
            features[f'{col}_diff_mean'] = diff.mean()
            features[f'{col}_diff_std'] = diff.std()
            features[f'{col}_diff_max'] = diff.abs().max()
    
    # Number of observations
    features['n_observations'] = len(group)
    
    return pd.Series(features)

print("Extracting features from training data...")
X_train = train_df.groupby('maneuver_Id').apply(extract_features).reset_index()
y_train = train_df.drop_duplicates('maneuver_Id')[['maneuver_Id', 'label']].set_index('maneuver_Id').loc[X_train['maneuver_Id'], 'label'].values

print(f"Features extracted: {X_train.shape}")
print(f"Feature names: {X_train.columns[1:].tolist()[:10]}...")

In [ ]:
# Handle NaN values in features
X_train = X_train.fillna(0)
print(f"Missing values after fillna: {X_train.isnull().sum().sum()}")
print(f"\nFeature statistics:\n{X_train.iloc[:, 1:].describe()}")

In [ ]:
# Extract features for test data
print("Extracting features from test data...")
X_test = test_df.groupby('maneuver_Id').apply(extract_features).reset_index()
X_test = X_test.fillna(0)
print(f"Test features extracted: {X_test.shape}")

## 5. Feature Selection & Scaling

In [ ]:
# Drop maneuver_Id and prepare for modeling
feature_cols = X_train.columns[1:].tolist()
X_train_features = X_train[feature_cols]
X_test_features = X_test[feature_cols]

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_features)
X_test_scaled = scaler.transform(X_test_features)

print(f"Training set shape: {X_train_scaled.shape}")
print(f"Test set shape: {X_test_scaled.shape}")

## 6. Model Training & Cross-Validation

In [ ]:
# Define models
models = {
    'XGBoost': xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    ),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        random_state=42
    )
}

# Cross-validation setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

In [ ]:
# Train and evaluate each model
for model_name, model in models.items():
    print(f"\nTraining {model_name}...")
    
    cv_scores = cross_validate(
        model, X_train_scaled, y_train,
        cv=skf,
        scoring=['accuracy', 'f1_weighted', 'f1_macro'],
        return_train_score=True,
        n_jobs=-1
    )
    
    cv_results[model_name] = cv_scores
    
    print(f"  Accuracy: {cv_scores['test_accuracy'].mean():.4f} (+/- {cv_scores['test_accuracy'].std():.4f})")
    print(f"  F1 (weighted): {cv_scores['test_f1_weighted'].mean():.4f} (+/- {cv_scores['test_f1_weighted'].std():.4f})")
    print(f"  F1 (macro): {cv_scores['test_f1_macro'].mean():.4f} (+/- {cv_scores['test_f1_macro'].std():.4f})")

In [ ]:
# Train final model on full training set
best_model_name = 'XGBoost'  # Can be updated based on CV results
final_model = models[best_model_name]
print(f"Training final {best_model_name} model on full training set...")
final_model.fit(X_train_scaled, y_train)
print("Done!")

## 7. Evaluation on Training Set

In [ ]:
# Predictions on training set
y_pred_train = final_model.predict(X_train_scaled)
y_pred_proba_train = final_model.predict_proba(X_train_scaled)

# Classification report
print("Classification Report (Training Set):\n")
print(classification_report(y_train, y_pred_train, target_names=['Class 0', 'Class 1', 'Class 2']))

In [ ]:
# Per-class F1 scores
precision, recall, f1, support = precision_recall_fscore_support(y_train, y_pred_train, average=None)
print("\nPer-Class Metrics:")
for i in range(len(np.unique(y_train))):
    print(f"Class {i}: Precision={precision[i]:.4f}, Recall={recall[i]:.4f}, F1={f1[i]:.4f}")
print(f"\nMinimum F1 (Final Score): {min(f1):.4f}")

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_train, y_pred_train)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Class 0', 'Class 1', 'Class 2'],
            yticklabels=['Class 0', 'Class 1', 'Class 2'])
plt.title('Confusion Matrix (Training Set)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 8. Feature Importance

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 20 Most Important Features:")
print(feature_importance.head(20))

In [ ]:
# Plot feature importance
plt.figure(figsize=(10, 8))
top_features = feature_importance.head(15)
plt.barh(range(len(top_features)), top_features['importance'].values)
plt.yticks(range(len(top_features)), top_features['feature'].values)
plt.xlabel('Importance')
plt.title('Top 15 Feature Importance')
plt.tight_layout()
plt.show()

## 9. Test Set Predictions

In [ ]:
# Make predictions on test set
y_pred_test = final_model.predict(X_test_scaled)

# Create submission file
submission = pd.DataFrame({
    'maneuver_id': X_test['maneuver_Id'],
    'class': y_pred_test
})

submission.to_csv('../output/submission.csv', index=False)
print(f"Submission saved! Shape: {submission.shape}")
print(f"\nClass distribution in submission:")
print(submission['class'].value_counts().sort_index())

In [ ]:
print("\n" + "="*50)
print("SUMMARY")
print("="*50)
print(f"Model: {best_model_name}")
print(f"Training Accuracy: {(y_pred_train == y_train).mean():.4f}")
print(f"Minimum F1 Score: {min(f1):.4f}")
print(f"Test predictions saved to: ../output/submission.csv")